<a href="https://colab.research.google.com/github/harishkulkarni10/ai-research-assistant-rag/blob/dev/notebooks/05_eval/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

In [ ]:
import shutil, os
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers.util import cos_sim

# Clone
# !git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")

bge_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
e5_path  = PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet"

chunks_bge = pd.read_parquet(bge_path)
chunks_e5  = pd.read_parquet(e5_path)

print(f"BGE: {len(chunks_bge):,} chunks")
print(f"E5:  {len(chunks_e5):,} chunks")
print(f"BGE columns: {list(chunks_bge.columns)}")

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the embedding models
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)
e5_model = SentenceTransformer("intfloat/e5-base-v2", device=device)

print("Models loaded successfully")

In [ ]:
import torch
import numpy as np

# Prepare embeddings as float32 tensors
bge_emb = torch.tensor(np.array(chunks_bge['embedding'].tolist(), dtype=np.float32))
e5_emb  = torch.tensor(np.array(chunks_e5['embedding'].tolist(), dtype=np.float32))

device = "cuda" if torch.cuda.is_available() else "cpu"
bge_emb = bge_emb.to(device)
e5_emb  = e5_emb.to(device)

def retrieve_bge(query: str, top_k: int = 5):
    q_emb = bge_model.encode(query, normalize_embeddings=True)
    q_emb = torch.tensor(q_emb, dtype=torch.float32).to(device).unsqueeze(0)
    scores = torch.mm(q_emb, bge_emb.T)[0]  # cosine sim (normalized)
    top_indices = torch.topk(scores, top_k).indices.cpu().numpy()
    return chunks_bge.iloc[top_indices].assign(score=scores[top_indices].cpu().numpy())

def retrieve_e5(query: str, top_k: int = 5):
    q_emb = e5_model.encode("query: " + query, normalize_embeddings=True)
    q_emb = torch.tensor(q_emb, dtype=torch.float32).to(device).unsqueeze(0)
    scores = torch.mm(q_emb, e5_emb.T)[0]
    top_indices = torch.topk(scores, top_k).indices.cpu().numpy()
    return chunks_e5.iloc[top_indices].assign(score=scores[top_indices].cpu().numpy())

print("Retrieval functions ready")

In [ ]:
queries = [
    "How are nonlinear dynamical systems modeled using differential equations?",
    "What methods are used to analyze stability and phase transitions in physical systems?",
    "How are time series and temporal dynamics analyzed in complex systems?",
    "What mathematical techniques are used for solving elliptic and partial differential equations?",
    "How are statistical distributions and energy models used in astrophysical systems?"
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*100}")
    print(f"Query {i}/{len(queries)}: {query}")
    print(f"{'='*100}\n")

    # BGE
    print("=== BGE (Exact) ===")
    bge_res = retrieve_bge(query, top_k=5)
    display(bge_res[["chunk_id", "paper_id", "score", "text"]])

    # E5
    print("\n=== E5 (Exact) ===")
    e5_res = retrieve_e5(query, top_k=5)
    display(e5_res[["chunk_id", "paper_id", "score", "text"]])

    # Previews
    print("\nBGE Top Preview:")
    print(bge_res.iloc[0]["text"][:1000] + "...")
    print("\nE5 Top Preview:")
    print(e5_res.iloc[0]["text"][:1000] + "...")
